# 👁️ 08 — Visão com domain randomization (Fase 2): candidato sintético

> **Célula única — `Runtime → Run all`.** Use **GPU** (Colab). `HF_REPO_ID` ASCII no formato `owner/repo` é obrigatório; a publicação exige também `POKER_PUBLISH_ARTIFACTS=1` e `HF_TOKEN`. O padrão é não publicar.

## O desafio
O objetivo é ler cartas, participantes e posição em layouts variados. O conjunto desta etapa é inteiramente sintético: ele mede generalização entre estilos do gerador, mas **não demonstra** generalização para clientes reais. O modelo continua sendo candidato experimental até passar por um corpus real, independente, agrupado por cliente/sessão/tema.

## A estratégia (verificada por double-check adversarial)
- **Síntese + domain randomization no eixo ESTILO-DE-UI** é uma hipótese de melhoria: gera milhares de mesas variando fonte do rank, cor do naipe, **deck 2-cores E 4-cores**, feltro, tema, escala, posição, blur/ruído, jogadores e dealer button.
- **YOLO11n** (nano) localiza + classifica **54 classes**: as 52 cartas + `seat` (jogador) + `button` (dealer). Naipes desenhados como **formas** (não glifos) → renderizam sempre + variedade.
- **Posição oficial por geometria:** dos assentos + botão detectados, ordena as cadeiras no sentido **horário** (ordem de ação), acha o botão e o herói e calcula a posição — a mesma `derive_position` que o backend usa. O número de `seat` detectados é a **contagem de participantes**.
- **Estresse sintético:** o gerador inclui **oclusão**, negativos (8% sem cartas) e augmentation. Isso testa falhas simuladas, não substitui validação real.
- **Gate interno:** avalia 20 estilos sintéticos held-out, disjuntos do treino. Essa métrica só autoriza avançar ao teste real; não prova ausência de viés nem prontidão de produção.

## Depois de rodar
O `.onnx` só deve substituir o baseline após avaliação cega em screenshots reais autorizados, com métricas por cliente/deck/tema, calibração, taxa de abstenção e paridade ONNX. Não há evidência para prometer que menos de 200 imagens seja suficiente.

*Números / OCR: para ler o pote/stacks use PaddleOCR (a F1 usa dígitos por template; o número é o elo fraco conhecido). As CARTAS, JOGADORES e o BOTÃO são o núcleo aqui.*


In [ ]:
# ============================================================
# VISAO COM DOMAIN RANDOMIZATION (Fase 2) - candidato sintetico
# Celula UNICA: Run all. GPU (Colab). Publicacao e opt-in explicito.
#
# A ESTRATEGIA (verificada por double-check adversarial):
#  - o gargalo NAO e ler a carta, e LOCALIZAR numa UI nunca vista;
#  - testa SINTESE + DOMAIN RANDOMIZATION no eixo ESTILO-DE-UI;
#  - treina YOLO11n (nano) pra LOCALIZAR+CLASSIFICAR 52 cartas + JOGADOR + BOTAO (54 classes);
#  - dos assentos+botao sai o Nº DE PARTICIPANTES e a POSICAO do heroi (regra oficial);
#  - mede em HELD-OUT sintetico; validacao real independente continua obrigatoria.
# ============================================================
import glob, hashlib, json, math, os, platform, random, re, shutil, subprocess, time
from pathlib import Path
ULTRALYTICS_VERSION = '8.3.223'
PILLOW_VERSION = '12.3.0'
NUMPY_VERSION = '2.3.1'
get_ipython().system(
    f'pip -q install ultralytics=={ULTRALYTICS_VERSION} '
    f'pillow=={PILLOW_VERSION} numpy=={NUMPY_VERSION}'
)

try:
    from google.colab import userdata
except ImportError:
    userdata = None
HF_REPO_ID = (userdata.get('HF_REPO_ID') if userdata is not None
              else os.environ.get('HF_REPO_ID'))
HF_REPO_ID_PATTERN = r'[A-Za-z0-9](?:[A-Za-z0-9._-]{0,94}[A-Za-z0-9])?/[A-Za-z0-9](?:[A-Za-z0-9._-]{0,94}[A-Za-z0-9])?'
if (not isinstance(HF_REPO_ID, str) or not HF_REPO_ID.isascii()
        or len(HF_REPO_ID) > 96 or re.fullmatch(HF_REPO_ID_PATTERN, HF_REPO_ID) is None
        or '..' in HF_REPO_ID or '--' in HF_REPO_ID):
    raise RuntimeError('HF_REPO_ID deve ser ASCII owner/repo canonico.')

import numpy as np
import PIL
import torch
import ultralytics
from PIL import Image, ImageDraw, ImageFont, ImageFilter

# ---------- logs estilo Use a Cabeca ----------
def box(t, ls=()):
    w = max([len(t)] + [len(x) for x in ls]) if ls else len(t)
    print("\n  +" + "-"*(w+2) + "+"); print("  | " + t.ljust(w) + " |"); print("  +" + "-"*(w+2) + "+")
    for x in ls: print("  | " + x.ljust(w) + " |")
    if ls: print("  +" + "-"*(w+2) + "+")

# ---------- CONFIG ----------
N_TRAIN, N_VAL = 6000, 900
EPOCHS, IMGSZ = 60, 640
SEED = 20260717
ARCH = "yolo11n"
PUBLISH_ARTIFACTS = os.environ.get("POKER_PUBLISH_ARTIFACTS") == "1"
ROOT = Path("/content/pokervision_run_20260717")
BASE_WEIGHTS_PATH = Path(os.environ.get('POKER_YOLO_BASE_PATH', ''))
BASE_WEIGHTS_SHA256 = os.environ.get('POKER_YOLO_BASE_SHA256', '')
if (not BASE_WEIGHTS_PATH.is_file()
        or re.fullmatch(r'[0-9a-f]{64}', BASE_WEIGHTS_SHA256) is None
        or hashlib.sha256(BASE_WEIGHTS_PATH.read_bytes()).hexdigest() != BASE_WEIGHTS_SHA256):
    raise RuntimeError('forneca peso base local com SHA-256 lowercase verificado.')
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
RANKS = "23456789TJQKA"; SUITS = "shdc"
CLASSES = [r+s for r in RANKS for s in SUITS]          # 52 classes: 'As','2h',...
SEAT, BUTTON = 52, 53                                   # +2 classes: jogador e dealer button
NAMES = CLASSES + ["seat", "button"]; NC = len(NAMES)  # 54 classes no total
RANK_DISP = {"T": "10"}

# ---------- posicao oficial a partir da geometria dos assentos (ordem de acao = horario) ----------
POS = {2:["SB","BB"], 3:["BTN","SB","BB"], 4:["BTN","SB","BB","UTG"],
       5:["BTN","SB","BB","UTG","HJ"], 6:["BTN","SB","BB","UTG","DJ","HJ"],
       7:["BTN","SB","BB","UTG","MP2","DJ","HJ"], 8:["BTN","SB","BB","UTG","UTG+1","MP2","DJ","HJ"],
       9:["BTN","SB","BB","UTG","UTG+1","MP1","MP2","DJ","HJ"]}
def derive_position(seat_centers, hero_xy, button_xy):
    """(nº de jogadores, posicao do heroi). Ordena assentos no sentido HORARIO ao redor
    da mesa; botao = offset 0, proximo horario = SB, etc. (convencao oficial de poker)."""
    n = len(seat_centers)
    if n < 2: return n, ""
    cxm = sum(p[0] for p in seat_centers)/n; cym = sum(p[1] for p in seat_centers)/n
    order = sorted(seat_centers, key=lambda p: math.atan2(p[1]-cym, p[0]-cxm))
    near = lambda t: min(range(n), key=lambda i: (order[i][0]-t[0])**2 + (order[i][1]-t[1])**2)
    hero_i = near(hero_xy)
    if button_xy is None or n not in POS: return n, ""
    off = (hero_i - near(button_xy)) % n
    return n, (POS[n][off] if off < len(POS[n]) else "")

box("VISAO COM DOMAIN RANDOMIZATION (F2) - o plano", [
    "1. gerar mesas 2D SINTETICAS com dezenas de ESTILOS (fonte/cor/deck/feltro/tema)",
    "2. desenhar JOGADORES (2-9) + dealer button -> aprende a CONTAR e achar o botao",
    f"3. treinar {ARCH}: 52 cartas + jogador + botao (54 classes) + oclusao/negativos",
    "4. dos assentos+botao: Nº de participantes e POSICAO do heroi (regra oficial)",
    "5. HELD-OUT sintetico: gate interno antes de uma avaliacao real independente",
])

# ---------- fontes variadas (domain randomization no estilo do rank) ----------
get_ipython().system('apt-get -qq install -y fonts-liberation fonts-dejavu-core fonts-freefont-ttf >/dev/null 2>&1')
FONTS = glob.glob("/usr/share/fonts/truetype/**/*.ttf", recursive=True)
FONTS = [f for f in FONTS if not any(b in f.lower() for b in ("emoji","symbol","mono"))][:14] or [None]
print(f"fontes p/ variar o rank: {len(FONTS)}")

# ---------- naipes desenhados (formas, nao glifos) => renderizam sempre + variedade ----------
def draw_suit(d, suit, cx, cy, s, color):
    if suit == "d":  # diamante
        d.polygon([(cx,cy-s),(cx+s*0.7,cy),(cx,cy+s),(cx-s*0.7,cy)], fill=color)
    elif suit == "h":  # coracao
        d.pieslice([cx-s,cy-s,cx,cy], 0, 360, fill=color)
        d.pieslice([cx,cy-s,cx+s,cy], 0, 360, fill=color)
        d.polygon([(cx-s*0.98,cy-s*0.1),(cx+s*0.98,cy-s*0.1),(cx,cy+s)], fill=color)
    elif suit == "s":  # espada
        d.polygon([(cx,cy-s),(cx+s*0.85,cy+s*0.4),(cx-s*0.85,cy+s*0.4)], fill=color)
        d.pieslice([cx-s*0.85,cy-s*0.1,cx,cy+s*0.6], 0, 360, fill=color)
        d.pieslice([cx,cy-s*0.1,cx+s*0.85,cy+s*0.6], 0, 360, fill=color)
        d.rectangle([cx-s*0.14,cy+s*0.2,cx+s*0.14,cy+s], fill=color)
        d.polygon([(cx-s*0.4,cy+s),(cx+s*0.4,cy+s),(cx,cy+s*0.5)], fill=color)
    else:  # club (paus)
        r = s*0.5
        d.ellipse([cx-r,cy-s*0.9,cx+r,cy+r*0.2], fill=color)
        d.ellipse([cx-s*0.9,cy-r*0.3,cx-s*0.9+2*r,cy-r*0.3+2*r], fill=color)
        d.ellipse([cx+s*0.9-2*r,cy-r*0.3,cx+s*0.9,cy-r*0.3+2*r], fill=color)
        d.rectangle([cx-s*0.14,cy,cx+s*0.14,cy+s], fill=color)
        d.polygon([(cx-s*0.4,cy+s),(cx+s*0.4,cy+s),(cx,cy+s*0.5)], fill=color)

def rand_color(rng, kind):
    j = lambda: rng.randint(-25, 25)
    base = {"black": (25,25,25), "red": (200,30,40), "blue": (30,90,210), "green": (20,120,40)}[kind]
    return tuple(int(np.clip(c + j(), 0, 255)) for c in base)

def render_card(rng, rank, suit, w):
    h = int(w*1.4); img = Image.new("RGB", (w, h), (0,0,0,0))
    bg = tuple(rng.randint(238,255) for _ in range(3))
    d = ImageDraw.Draw(img)
    d.rounded_rectangle([0,0,w-1,h-1], radius=rng.randint(3,w//8), fill=bg,
                        outline=tuple(rng.randint(150,200) for _ in range(3)), width=rng.randint(1,2))
    four = rng.random() < 0.4  # 40% deck 4-cores
    cmap = {"s":"black","c":("green" if four else "black"),"h":"red","d":("blue" if four else "red")}
    col = rand_color(rng, cmap[suit])
    try: f = ImageFont.truetype(rng.choice(FONTS), int(w*rng.uniform(0.30,0.40)))
    except Exception: f = ImageFont.load_default()
    d.text((w*0.08, h*0.03), RANK_DISP.get(rank, rank), font=f, fill=col)
    draw_suit(d, suit, int(w*0.24), int(h*0.30), w*0.11, col)          # naipe do indice
    draw_suit(d, suit, w//2, int(h*0.62), w*rng.uniform(0.22,0.28), col)  # naipe central
    return img

def draw_players(d, rng, srng, W, H, labels):
    """Desenha 2-9 jogadores ao redor da mesa (avatar de cor/forma QUALQUER = domain
    randomization: o modelo aprende 'jogador', nao uma cor fixa). Devolve (centros, botao)."""
    n = rng.randint(2, 9)
    cx0, cy0, rx, ry = W/2, H/2, W*0.40, H*0.44
    seats = [(cx0 + rx*math.cos(math.radians(90 + i*360/n)),
              cy0 + ry*math.sin(math.radians(90 + i*360/n))) for i in range(n)]
    for (sx, sy) in seats:
        rr = srng.randint(20, 34)
        acol = tuple(srng.randint(20, 235) for _ in range(3))   # avatar de cor arbitraria
        if srng.random() < 0.5:
            d.ellipse([sx-rr, sy-rr, sx+rr, sy+rr], fill=acol, outline=(235,235,235), width=2)
        else:
            d.rounded_rectangle([sx-rr, sy-rr, sx+rr, sy+rr], radius=srng.randint(4,rr), fill=acol)
        pw, ph = int(rr*2.1), int(rr*0.7)                       # plaqueta de stack (escura)
        d.rounded_rectangle([sx-pw/2, sy+rr+2, sx+pw/2, sy+rr+2+ph], radius=4, fill=(28,30,36))
        labels.append((SEAT, sx/W, sy/H, (2*rr)/W, (2*rr)/H))
    return seats, rng.randrange(n)

def gen_sample(rng, styles_seed):
    """Uma mesa 2D + labels YOLO. styles_seed fixa o 'estilo' (fundo/escala) do conjunto."""
    W, H = 900, 600
    srng = random.Random(styles_seed)
    felt = tuple(srng.randint(15,110) for _ in range(3))
    img = Image.new("RGB", (W, H), felt)
    d = ImageDraw.Draw(img)
    if srng.random() < 0.7:  # vinheta/tema
        d.ellipse([W*0.08,H*0.03,W*0.92,H*0.97], fill=tuple(min(255,c+srng.randint(8,30)) for c in felt))
    labels = []
    # jogadores ANTES das cartas (cartas por cima); botao depois (por cima de tudo)
    seats, bseat = draw_players(d, rng, srng, W, H, labels)
    deck = CLASSES[:]; rng.shuffle(deck)
    neg = rng.random() < 0.08  # NEGATIVO: mesa sem cartas visiveis (reduz falso-positivo)
    n_board = 0 if neg else rng.choice([0,3,3,4,5]); hole = [] if neg else deck[:2]; board = deck[2:2+n_board]
    cw = int(srng.uniform(40,84))  # baralhos de TAMANHOS diferentes por aluno
    def place(card, cx, cy):
        ci = render_card(rng, card[0], card[1], cw)
        img.paste(ci, (cx-ci.width//2, cy-ci.height//2), ci.convert("RGBA"))
        labels.append((CLASSES.index(card), cx/W, cy/H, ci.width/W, ci.height/H))
    if board:
        gap = int(cw*1.3); x0 = W//2 - (len(board)-1)*gap//2; by = H//2 + rng.randint(-25,10)
        for i,c in enumerate(board): place(c, x0+i*gap+rng.randint(-4,4), by)
    hx = W//2 + rng.randint(-50,50); hy = int(H*0.76)+rng.randint(-12,10)  # acima do avatar do heroi
    for i,c in enumerate(hole): place(c, hx-46+i*92, hy)
    # dealer button AO LADO do assento sorteado (radial leve + tangencial) e POR CIMA das cartas
    sxb, syb = seats[bseat]
    ux, uy = (W/2-sxb), (H/2-syb); lg = math.hypot(ux,uy) or 1.0; ux, uy = ux/lg, uy/lg
    bx = sxb + ux*16 - uy*42; by2 = syb + uy*16 + ux*42
    brr = srng.randint(10, 16)
    bcol = tuple((srng.randint(160,255) if k<2 else srng.randint(0,110)) for k in range(3))  # tende a dourado
    d.ellipse([bx-brr, by2-brr, bx+brr, by2+brr], fill=bcol, outline=(120,95,10), width=2)
    labels.append((BUTTON, bx/W, by2/H, (2*brr)/W, (2*brr)/H))
    if rng.random() < 0.30:  # OCLUSAO: fichas/mao sobre parte das cartas (robustez real)
        for _ in range(rng.randint(1, 3)):
            ox, oy = rng.randint(0, W), rng.randint(int(H*0.42), H)
            ow, oh = rng.randint(18, 64), rng.randint(12, 44)
            d.rounded_rectangle([ox, oy, ox+ow, oy+oh], radius=6, fill=tuple(rng.randint(20, 215) for _ in range(3)))
    if rng.random() < 0.5:  # ruido/blur leve as vezes
        img = img.filter(ImageFilter.GaussianBlur(rng.uniform(0,0.8)))
    arr = np.asarray(img).astype("int16") + np.random.default_rng(styles_seed+rng.randint(0,9999)).normal(0, rng.uniform(0,6), (H,W,3))
    return Image.fromarray(np.clip(arr,0,255).astype("uint8")), labels

# ---------- gera o dataset (train com estilos 0..79 ; val = HELD-OUT 80..99) ----------
box("[1/3] Gerando dataset sintetico (estilos de TREINO x HELD-OUT disjuntos)")
for split in ("train","val"):
    os.makedirs(f"{ROOT}/images/{split}", exist_ok=True); os.makedirs(f"{ROOT}/labels/{split}", exist_ok=True)
def build(split, n, style_ids):
    split_seed = {"train": SEED, "val": SEED + 1}[split]
    rng = random.Random(split_seed)
    for i in range(n):
        img, labels = gen_sample(rng, rng.choice(style_ids))
        img.save(f"{ROOT}/images/{split}/{i:05}.jpg", quality=88)
        with open(f"{ROOT}/labels/{split}/{i:05}.txt","w") as fh:
            fh.write("\n".join(f"{c} {x:.5f} {y:.5f} {w:.5f} {h:.5f}" for c,x,y,w,h in labels))
        if i % 1000 == 0: print(f"  {split}: {i}/{n}")
build("train", N_TRAIN, list(range(0,80)))       # 80 estilos de treino
build("val",   N_VAL,   list(range(80,100)))     # 20 estilos NUNCA vistos (held-out)
with open(f"{ROOT}/data.yaml","w") as fh:
    fh.write(f"path: {ROOT}\ntrain: images/train\nval: images/val\nnc: {NC}\nnames: {NAMES}\n")
print(f"dataset pronto. classes: {NC} (52 cartas + jogador + botao)")

# ---------- treina a arquitetura declarada ----------
box(f"[2/3] Treinando {ARCH} (52 cartas + jogador + botao = 54 classes)")
from ultralytics import YOLO
model = YOLO(str(BASE_WEIGHTS_PATH))
model.train(data=f"{ROOT}/data.yaml", epochs=EPOCHS, imgsz=IMGSZ, batch=32,
            patience=15, device=0, project=ROOT, name="cards", verbose=True,
            seed=SEED, deterministic=True,
            mosaic=1.0, mixup=0.1, copy_paste=0.1)

# ---------- avalia no HELD-OUT + exporta ----------
box("[3/3] HELD-OUT (estilos nunca vistos) + export ONNX")
m = model.val(data=f"{ROOT}/data.yaml", split="val")
mAP50 = float(m.box.map50); mAP = float(m.box.map)
onnx = model.export(format="onnx", imgsz=IMGSZ, opset=12)
onnx_path = Path(onnx)
pt_path = Path(model.trainer.best)
if not onnx_path.is_file() or not pt_path.is_file():
    raise RuntimeError("export incompleto: ONNX e best.pt sao obrigatorios")
sha256 = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
onnx_sha256, pt_sha256 = sha256(onnx_path), sha256(pt_path)
candidate_dir = ROOT / 'candidates' / onnx_sha256
candidate_dir.mkdir(parents=True)
candidate_onnx = candidate_dir / f'table_{ARCH}.onnx'
candidate_pt = candidate_dir / f'table_{ARCH}.pt'
shutil.copy2(onnx_path, candidate_onnx)
shutil.copy2(pt_path, candidate_pt)
dataset_files = sorted(
    [*ROOT.glob('images/**/*.jpg'), *ROOT.glob('labels/**/*.txt')],
    key=lambda p: p.as_posix(),
)
dataset_inventory = [
    {'path': p.relative_to(ROOT).as_posix(), 'sha256': sha256(p)} for p in dataset_files
]
dataset_inventory_sha256 = hashlib.sha256(
    json.dumps(dataset_inventory, sort_keys=True, separators=(',', ':')).encode(),
).hexdigest()
manifest = {
    "schema_version": 1, "architecture": ARCH, "seed": SEED,
    "environment": {
        "python": platform.python_version(), "torch": torch.__version__,
        "cuda": torch.version.cuda, "ultralytics": ultralytics.__version__,
        "numpy": np.__version__, "pillow": PIL.__version__,
    },
    "base_weights_sha256": BASE_WEIGHTS_SHA256,
    "training_data": "synthetic-only", "real_world_validated": False,
    "dataset_inventory_sha256": dataset_inventory_sha256,
    "n_train": N_TRAIN, "n_val": N_VAL, "imgsz": IMGSZ,
    "metrics_synthetic_heldout": {"map50": mAP50, "map50_95": mAP},
    "artifacts": {"onnx_sha256": onnx_sha256, "pt_sha256": pt_sha256},
}
manifest_path = candidate_dir / 'manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")

# ---------- DEMO: da imagem -> jogadores + POSICAO do heroi (imagens held-out) ----------
box("DEMO: contar participantes + derivar posicao (estilos NUNCA vistos)")
def read_players(res, W, H):
    """Extrai assentos/botao/heroi das deteccoes YOLO e deriva (nº jogadores, posicao)."""
    seats, button, hole_pts = [], None, []
    for b in res.boxes:
        cls = int(b.cls[0]); x, y, w, h = b.xywh[0].tolist()
        if cls == SEAT: seats.append((x, y))
        elif cls == BUTTON: button = (x, y)
        elif cls < 52 and y > H*0.6: hole_pts.append((x, y))   # cartas do heroi (embaixo)
    hero = (sum(p[0] for p in hole_pts)/len(hole_pts), sum(p[1] for p in hole_pts)/len(hole_pts)) \
        if hole_pts else (W/2, H*0.9)
    return derive_position(seats, hero, button)
for f in sorted(glob.glob(f"{ROOT}/images/val/*.jpg"))[:5]:
    im = Image.open(f); r = model.predict(f, imgsz=IMGSZ, verbose=False)[0]
    npl, pos = read_players(r, im.width, im.height)
    print(f"  {os.path.basename(f)}: {npl} jogadores, heroi em {pos or '?'}")

box("RESULTADO DO GATE SINTETICO", [
    f"mAP50 em ESTILOS NUNCA VISTOS: {mAP50*100:.1f}%   (localizacao+classe corretas)",
    f"mAP50-95: {mAP*100:.1f}%",
    "54 classes: 52 cartas + jogador + botao -> conta participantes E acha a posicao.",
    "Resultado restrito aos estilos do gerador; nao comprova desempenho em UIs reais.",
    "Proximo gate: corpus real autorizado, independente e agrupado por cliente/sessao/tema.",
])
print("ONNX:", onnx)

# ---------- publica no HF somente com opt-in explicito ----------
if PUBLISH_ARTIFACTS:
    tok = (userdata.get("HF_TOKEN") if userdata is not None
           else os.environ.get("HF_TOKEN"))
    if not tok:
        raise RuntimeError("POKER_PUBLISH_ARTIFACTS=1 exige HF_TOKEN")
    from huggingface_hub import HfApi
    api = HfApi(token=tok)
    api.create_repo(HF_REPO_ID, repo_type="model", exist_ok=True, private=True)
    remote_dir = f'candidates/{onnx_sha256}'
    publish_commits = {}
    for local in (candidate_onnx, candidate_pt, manifest_path):
        commit = api.upload_file(
            path_or_fileobj=str(local), path_in_repo=f'{remote_dir}/{local.name}',
            repo_id=HF_REPO_ID, repo_type="model",
        )
        if re.fullmatch(r'[0-9a-f]{40}', commit.oid) is None:
            raise RuntimeError('Hub nao retornou commit imutavel verificavel.')
        publish_commits[local.name] = commit.oid
    publish_receipt = {
        'remote_dir': remote_dir, 'commits': publish_commits,
        'artifact_sha256': onnx_sha256, 'canonical_updated': False,
    }
    (candidate_dir / 'hf_publish_receipt.json').write_text(
        json.dumps(publish_receipt, indent=2, sort_keys=True), encoding='utf-8',
    )
    print("candidato publicado em", remote_dir, "; canonico inalterado")
else:
    print("HF nao publicado (padrao seguro; defina POKER_PUBLISH_ARTIFACTS=1 para habilitar).")
print("\nFIM. Candidato exportado; validacao real independente permanece obrigatoria.")